In [ ]:
!nvidia-smi

Sun Dec  7 20:22:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [ ]:
import cupy as cp
print(cp.cuda.runtime.getDeviceCount(), "GPU(s) detected")

1 GPU(s) detected


In [ ]:
import os

print("Exists:", os.path.exists("TMDB_movie_dataset_v112.csv"))

Exists: True


In [ ]:
import cupy as cp
cp.show_config()

OS                           : Linux-6.6.105+-x86_64-with-glibc2.35
Python Version               : 3.12.12
CuPy Version                 : 13.6.0
CuPy Platform                : NVIDIA CUDA
NumPy Version                : 2.0.2
SciPy Version                : 1.16.3
Cython Build Version         : 3.0.12
Cython Runtime Version       : 3.0.12
CUDA Root                    : /usr/local/cuda
nvcc PATH                    : /usr/local/cuda/bin/nvcc
CUDA Build Version           : 12090
CUDA Driver Version          : 12040
CUDA Runtime Version         : 12090 (linked to CuPy) / 12060 (locally installed)
CUDA Extra Include Dirs      : ['/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/include']
cuBLAS Version               : (available)
cuFFT Version                : 11300
cuRAND Version               : 10307
cuSOLVER Version             : (11, 6, 3)
cuSPARSE Version             : (available)
NVRTC Version                : (12, 6)
Thrust Version               : 200800
CUB Build Version   

In [ ]:
!pip install numba cuda-python -q

In [ ]:
!pip install cupy-cuda12x numba pandas

In [ ]:
!pip install cupy-cuda12x

In [ ]:
import csv
import time
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda   # 👈 جديد
import math              # 👈 جديد


#Serial code
## whole dataset size


In [ ]:
import csv
import time
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

MAX_LINE   = 65536
MAX_FIELDS = 64
MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"


def trim_tail_only(s: str) -> str:
    return s.rstrip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        if s[i:i+4].isdigit():
            y = int(s[i:i+4])
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def insert_top_py(arr, title, year, rev, pop, score, max_top=MAX_TOP):
    if score <= 0:
        return

    n = len(arr)

    if n < max_top:
        pos = n
        arr.append(None)
        while pos > 0 and score > arr[pos-1]["score"]:
            arr[pos] = arr[pos-1]
            pos -= 1
        arr[pos] = {
            "title": title,
            "year": year,
            "revenue": rev,
            "popularity": pop,
            "score": score,
        }
        return

    if score <= arr[-1]["score"]:
        return

    pos = max_top - 1
    while pos > 0 and score > arr[pos-1]["score"]:
        arr[pos] = arr[pos-1]
        pos -= 1

    arr[pos] = {
        "title": title,
        "year": year,
        "revenue": rev,
        "popularity": pop,
        "score": score,
    }


def main():
    print(f"Using dataset: {DATASET_PATH}")

    total_start = time.perf_counter()
    cpu_start = time.perf_counter()

    total_movies = 0

    sum_year = np.zeros(YEAR_RANGE, dtype=np.float64)
    count_year = np.zeros(YEAR_RANGE, dtype=np.int64)

    gpu_year_idx_list = []
    gpu_rating_list = []

    genre_names = []
    genre_sum = []
    genre_cnt = []

    top_rev = []
    top_pop = []

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)

        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]

            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])

            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            if FIRST_YEAR <= year <= LAST_YEAR and rating > 0:
                idx = year - FIRST_YEAR
                sum_year[idx] += rating
                count_year[idx] += 1

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)

                parts = tmp.split(",")
                first = parts[0] if parts else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)

                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()

                    found = -1
                    for g_idx, g_name in enumerate(genre_names):
                        if g_name.lower() == first.lower():
                            found = g_idx
                            break

                    if found == -1 and len(genre_names) < MAX_GENRES:
                        genre_names.append(first)
                        genre_sum.append(0.0)
                        genre_cnt.append(0)
                        found = len(genre_names) - 1

                    if found != -1:
                        genre_sum[found] += rating
                        genre_cnt[found] += 1

            if rev > 0.0 and title:
                insert_top_py(top_rev, title, year, rev, pop, rev)
            if pop > 0.0 and title:
                insert_top_py(top_pop, title, year, rev, pop, pop)

            if rating > 0.0 and FIRST_YEAR <= year <= LAST_YEAR:
                gpu_year_idx_list.append(year - FIRST_YEAR)
                gpu_rating_list.append(rating)

    cpu_end = time.perf_counter()

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and gpu_year_idx_list:
        h_year_idx = np.array(gpu_year_idx_list, dtype=np.int32)
        h_rating = np.array(gpu_rating_list, dtype=np.float64)

        gpu_total_transfer_start = time.perf_counter()
        d_year_idx = cp.asarray(h_year_idx)
        d_rating = cp.asarray(h_rating)

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = d_year_idx >= 0
        year_valid = d_year_idx[valid_mask]
        rating_valid = d_rating[valid_mask]

        d_sum_year = cp.bincount(
            year_valid,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_valid,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        _ = cp.asnumpy(d_sum_year)
        _ = cp.asnumpy(d_count_year)
        gpu_total_transfer_end = time.perf_counter()

    print("=====================================")
    print("Serial Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for i, m in enumerate(top_rev[:MAX_TOP]):
        print(f"{i+1:2d}. \"{m['title']}\" | Revenue: {m['revenue']:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for i, m in enumerate(top_pop[:MAX_TOP]):
        print(f"{i+1:2d}. \"{m['title']}\" | Year:{m['year']} | Popularity: {m['popularity']:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        idx = y - FIRST_YEAR
        if count_year[idx] > 0:
            avg = sum_year[idx] / count_year[idx]
            print(f"{y:4d}  {count_year[idx]:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    for name, s, c in zip(genre_names, genre_sum, genre_cnt):
        avg = s / c if c > 0 else 0.0
        print(f"{name:<20s} {c:6d}   {avg:.3f}")

    total_end = time.perf_counter()
    total_time = total_end - total_start
    cpu_time = cpu_end - cpu_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + processing on CPU): {cpu_time:.6f} s")

    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available or no valid data).")

    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv
Serial Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 2. "babben: the movie" | Revenue: 4999999999.00
 3. "TikTok Rizz Party" | Revenue: 3000000000.00
 4. "Adventures in Bora Bora" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Year:2020 | Popularity: 2020.29
 4. "The Nun II" | Year:2023 | Popularity: 1692.78
 5. "Meg 2: The Trench" | Year:2023 | Popularity: 1567.27
 6. "Retribution" | Year:202

#parallel code
## whole dataset size

In [ ]:
import csv
import time
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv
GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Year:2020 | Popularity: 2020.29
 4. "The Nun II" | Year:2023 | Popularity: 1692.78
 5. "Meg 2: The Trench" | Year:2023 | Popularity: 1567.27
 6. "Retribution" | Year:2023 |

#Tunned parallel

#parallel
## 64 chunck size
##2 threads

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 64
BLOCK_SIZE_GRID = 64
BLOCKS_PER_GRID = 2


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 64
blocks_per_grid (auto from N)  = 16384
GPU kernel execution time (block tuning) = 0.158328 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 64
BLOCKS_PER_GRID (number of blocks)  = 2
GPU kernel execution time (grid tuning) = 0.096125 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Ye

#64 chunck size
##4 threads

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 64
BLOCK_SIZE_GRID = 64
BLOCKS_PER_GRID = 4


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 64
blocks_per_grid (auto from N)  = 16384
GPU kernel execution time (block tuning) = 0.279890 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE_GRID (threads_per_block) = 64
BLOCKS_PER_GRID (number of blocks)  = 4
GPU kernel execution time (grid tuning) = 0.101148 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endga

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


#64
##8 threads

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 64
BLOCK_SIZE_GRID = 64
BLOCKS_PER_GRID = 8


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 64
blocks_per_grid (auto from N)  = 16384
GPU kernel execution time (block tuning) = 0.167373 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 64
BLOCKS_PER_GRID (number of blocks)  = 8
GPU kernel execution time (grid tuning) = 0.205808 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Ye

#64
##16

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 64
BLOCK_SIZE_GRID = 64
BLOCKS_PER_GRID = 16


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 64
blocks_per_grid (auto from N)  = 16384
GPU kernel execution time (block tuning) = 0.186099 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 64
BLOCKS_PER_GRID (number of blocks)  = 16
GPU kernel execution time (grid tuning) = 0.101098 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#64
##20

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 64
BLOCK_SIZE_GRID = 64
BLOCKS_PER_GRID = 20


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 64
blocks_per_grid (auto from N)  = 16384
GPU kernel execution time (block tuning) = 0.208103 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE_GRID (threads_per_block) = 64
BLOCKS_PER_GRID (number of blocks)  = 20
GPU kernel execution time (grid tuning) = 0.190462 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]



/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 20 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Year:2020 | Popularity: 2020.29
 4. "The Nun II" | Year:2023 | Popularity: 1692.78
 5. "Meg 2: The Trench" | Year:2023 | Popularity: 1567.27
 6. "Retribution" | Year:2023 | Popularity: 1547.22
 7. "Talk to Me" | Year:2023 | 

#128
##2

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 128
BLOCK_SIZE_GRID = 128
BLOCKS_PER_GRID = 2


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 128
blocks_per_grid (auto from N)  = 8192
GPU kernel execution time (block tuning) = 0.173713 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 128
BLOCKS_PER_GRID (number of blocks)  = 2
GPU kernel execution time (grid tuning) = 0.095062 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#128
#4

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 128
BLOCK_SIZE_GRID = 128
BLOCKS_PER_GRID = 4


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 128
blocks_per_grid (auto from N)  = 8192
GPU kernel execution time (block tuning) = 0.163820 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 128
BLOCKS_PER_GRID (number of blocks)  = 4
GPU kernel execution time (grid tuning) = 0.209865 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#128
##6

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 128
BLOCK_SIZE_GRID = 128
BLOCKS_PER_GRID = 6


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 128
blocks_per_grid (auto from N)  = 8192
GPU kernel execution time (block tuning) = 0.172858 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 128
BLOCKS_PER_GRID (number of blocks)  = 8
GPU kernel execution time (grid tuning) = 0.099808 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y

#128
##8

In [ ]:
import csv
import time
import math
import numpy as np

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

from numba import cuda

MAX_TOP    = 10
MAX_GENRES = 200
FIRST_YEAR = 1800
LAST_YEAR  = 2025
YEAR_RANGE = (LAST_YEAR - FIRST_YEAR + 1)

DATASET_PATH = "/content/TMDB_movie_dataset_v112.csv"

BLOCK_SIZE = 128
BLOCK_SIZE_GRID = 128
BLOCKS_PER_GRID = 8


def trim_tail_only(s: str) -> str:
    return s.strip()


def clean_genre_c_style(s: str) -> str:
    if s is None:
        return ""
    temp_chars = []
    for ch in s:
        if ch not in ['"', '[', ']']:
            temp_chars.append(ch)
    tmp = "".join(temp_chars)
    tmp = trim_tail_only(tmp)
    return tmp


def parse_year_c_style(s: str) -> int:
    for i in range(len(s) - 3):
        seg = s[i:i+4]
        if seg.isdigit():
            y = int(seg)
            if FIRST_YEAR <= y <= LAST_YEAR:
                return y
    return 0


def to_double_c_style(s: str) -> float:
    try:
        return float(s)
    except ValueError:
        return 0.0


def gpu_block_tuning_experiment(ratings_np):
    print("\n=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for block-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)

    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(N / threads_per_block)

    @cuda.jit
    def scale_kernel(arr_in, arr_out, factor):
        i = cuda.grid(1)
        if i < arr_in.size:
            arr_out[i] = arr_in[i] * factor

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE (threads_per_block) = {threads_per_block}")
    print(f"blocks_per_grid (auto from N)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (block tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (block):", sample)
    print("==========================================\n")


def gpu_grid_tuning_experiment(ratings_np):
    print("\n=== GPU GRID-SIZE TUNING (Numba CUDA) ===")

    if not cuda.is_available():
        print("CUDA not available in this environment.")
        print("==========================================\n")
        return

    N = ratings_np.size
    if N == 0:
        print("No ratings available for grid-size tuning.")
        print("==========================================\n")
        return

    ratings_np32 = ratings_np.astype(np.float32)
    d_in = cuda.to_device(ratings_np32)
    d_out = cuda.device_array_like(d_in)

    threads_per_block = BLOCK_SIZE_GRID
    blocks_per_grid = BLOCKS_PER_GRID

    @cuda.jit
    def scale_kernel_gridstride(arr_in, arr_out, factor):
        i = cuda.grid(1)
        stride = cuda.gridsize(1)
        n = arr_in.size
        while i < n:
            arr_out[i] = arr_in[i] * factor
            i += stride

    cuda.synchronize()
    start = time.perf_counter()

    scale_kernel_gridstride[blocks_per_grid, threads_per_block](d_in, d_out, 1.5)

    cuda.synchronize()
    end = time.perf_counter()
    gpu_time = end - start

    print(f"BLOCK_SIZE_GRID (threads_per_block) = {threads_per_block}")
    print(f"BLOCKS_PER_GRID (number of blocks)  = {blocks_per_grid}")
    print(f"GPU kernel execution time (grid tuning) = {gpu_time:.6f} s")

    sample = d_out.copy_to_host()[:5]
    print("Sample scaled ratings (grid):", sample)
    print("==========================================\n")


def main():
    print(f"Using dataset: {DATASET_PATH}")
    total_start = time.perf_counter()

    cpu_start = time.perf_counter()

    titles = []
    years = []
    ratings = []
    revenues = []
    popularities = []

    genre_to_id = {}
    id_to_genre = []
    genre_ids = []
    genre_ratings = []

    total_movies = 0

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            print("Empty file.")
            return

        idxTitle = idxRating = idxRelease = idxRev = -1
        idxBudget = idxPop = idxGenre = -1

        for i, name in enumerate(header):
            col = trim_tail_only(name).lower()
            if col == "title":
                idxTitle = i
            elif col == "vote_average":
                idxRating = i
            elif col == "revenue":
                idxRev = i
            elif col == "budget":
                idxBudget = i
            elif col == "popularity":
                idxPop = i
            elif col == "release_date":
                idxRelease = i
            elif col == "genres":
                idxGenre = i

        num_fields = len(header)

        for row in reader:
            if len(row) < num_fields:
                continue

            total_movies += 1

            title = ""
            rating = rev = budget = pop = 0.0
            year = 0

            if idxTitle >= 0:
                title = trim_tail_only(row[idxTitle])[:255]
            if idxRating >= 0:
                rating = to_double_c_style(row[idxRating])
            if idxRev >= 0:
                rev = to_double_c_style(row[idxRev])
            if idxBudget >= 0:
                budget = to_double_c_style(row[idxBudget])
            if idxPop >= 0:
                pop = to_double_c_style(row[idxPop])
            if idxRelease >= 0:
                tmp = row[idxRelease][:63]
                tmp = trim_tail_only(tmp)
                year = parse_year_c_style(tmp)

            titles.append(title)
            years.append(year)
            ratings.append(rating)
            revenues.append(rev)
            popularities.append(pop)

            if idxGenre >= 0 and rating > 0:
                tmp = row[idxGenre][:255]
                tmp = clean_genre_c_style(tmp)
                first = tmp.split(",")[0] if tmp else ""
                first = clean_genre_c_style(first)
                first = trim_tail_only(first)
                if len(first) > 0:
                    first = first[0].upper() + first[1:].lower()
                    if first not in genre_to_id and len(genre_to_id) < MAX_GENRES:
                        gid = len(id_to_genre)
                        genre_to_id[first] = gid
                        id_to_genre.append(first)
                    gid = genre_to_id.get(first, None)
                    if gid is not None:
                        genre_ids.append(gid)
                        genre_ratings.append(rating)

    cpu_before_gpu_end = time.perf_counter()

    years_np   = np.array(years, dtype=np.int32)
    ratings_np = np.array(ratings, dtype=np.float64)
    revenues_np = np.array(revenues, dtype=np.float64)
    pops_np     = np.array(popularities, dtype=np.float64)

    num_genres = len(id_to_genre)
    genre_ids_np = np.array(genre_ids, dtype=np.int32) if num_genres > 0 else None
    genre_ratings_np = np.array(genre_ratings, dtype=np.float64) if num_genres > 0 else None

    gpu_block_tuning_experiment(ratings_np)
    gpu_grid_tuning_experiment(ratings_np)

    gpu_total_transfer_start = gpu_total_transfer_end = None
    gpu_compute_start = gpu_compute_end = None

    if CUPY_AVAILABLE and len(years_np) > 0:
        gpu_total_transfer_start = time.perf_counter()

        d_years = cp.asarray(years_np)
        d_ratings = cp.asarray(ratings_np)
        d_revenues = cp.asarray(revenues_np)
        d_pops = cp.asarray(pops_np)

        if num_genres > 0:
            d_genre_ids = cp.asarray(genre_ids_np)
            d_genre_ratings = cp.asarray(genre_ratings_np)
        else:
            d_genre_ids = None
            d_genre_ratings = None

        stream = cp.cuda.Stream.null
        stream.synchronize()
        gpu_compute_start = time.perf_counter()

        valid_mask = (d_ratings > 0) & (d_years >= FIRST_YEAR) & (d_years <= LAST_YEAR)
        year_idx = (d_years[valid_mask] - FIRST_YEAR).astype(cp.int32)
        rating_valid = d_ratings[valid_mask]

        d_sum_year = cp.bincount(
            year_idx,
            weights=rating_valid,
            minlength=YEAR_RANGE
        ).astype(cp.float64)

        d_count_year = cp.bincount(
            year_idx,
            minlength=YEAR_RANGE
        ).astype(cp.int64)

        if d_genre_ids is not None:
            d_sum_genre = cp.bincount(
                d_genre_ids,
                weights=d_genre_ratings,
                minlength=num_genres
            ).astype(cp.float64)

            d_count_genre = cp.bincount(
                d_genre_ids,
                minlength=num_genres
            ).astype(cp.int64)
        else:
            d_sum_genre = None
            d_count_genre = None

        valid_rev = d_revenues > 0
        rev_for_sort = d_revenues.copy()
        rev_for_sort[~valid_rev] = -1
        top_rev_idx_gpu = cp.argsort(rev_for_sort)[-MAX_TOP:][::-1]

        valid_pop = d_pops > 0
        pop_for_sort = d_pops.copy()
        pop_for_sort[~valid_pop] = -1
        top_pop_idx_gpu = cp.argsort(pop_for_sort)[-MAX_TOP:][::-1]

        stream.synchronize()
        gpu_compute_end = time.perf_counter()

        sum_year_gpu = cp.asnumpy(d_sum_year)
        count_year_gpu = cp.asnumpy(d_count_year)

        if d_sum_genre is not None:
            sum_genre_gpu = cp.asnumpy(d_sum_genre)
            count_genre_gpu = cp.asnumpy(d_count_genre)
        else:
            sum_genre_gpu = None
            count_genre_gpu = None

        top_rev_idx = cp.asnumpy(top_rev_idx_gpu).astype(int)
        top_pop_idx = cp.asnumpy(top_pop_idx_gpu).astype(int)

        gpu_total_transfer_end = time.perf_counter()

    else:
        sum_year_gpu = np.zeros(YEAR_RANGE, dtype=np.float64)
        count_year_gpu = np.zeros(YEAR_RANGE, dtype=np.int64)
        sum_genre_gpu = None
        count_genre_gpu = None
        top_rev_idx = np.argsort(revenues_np)[-MAX_TOP:][::-1]
        top_pop_idx = np.argsort(pops_np)[-MAX_TOP:][::-1]

    cpu_after_gpu_start = time.perf_counter()

    print("=====================================")
    print("GPU Version – TMDB Movie Analysis")
    print(f"Movies processed: {total_movies}\n")

    print(f"Top {MAX_TOP} Movies by Revenue:")
    for rank, idx in enumerate(top_rev_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Revenue: {revenues[idx]:.2f}")

    print(f"\nTop {MAX_TOP} Movies by Popularity:")
    for rank, idx in enumerate(top_pop_idx[:MAX_TOP], start=1):
        print(f"{rank:2d}. \"{titles[idx]}\" | Year:{years[idx]} | Popularity: {popularities[idx]:.2f}")

    print("\nAverage Rating per Year:")
    print("Year  Count  Avg")
    for y in range(FIRST_YEAR, LAST_YEAR + 1):
        i = y - FIRST_YEAR
        cnt = int(count_year_gpu[i])
        if cnt > 0:
            avg = sum_year_gpu[i] / cnt
            print(f"{y:4d}  {cnt:6d}  {avg:.3f}")

    print("\nAverage Rating per Genre:")
    print("Genre                 Count    Avg")
    if sum_genre_gpu is not None:
        for gid, name in enumerate(id_to_genre):
            cnt = int(count_genre_gpu[gid])
            if cnt > 0:
                avg = sum_genre_gpu[gid] / cnt
                print(f"{name:<20s} {cnt:6d}   {avg:.3f}")

    cpu_after_gpu_end = time.perf_counter()
    total_end = time.perf_counter()

    cpu_time = (cpu_before_gpu_end - cpu_start) + (cpu_after_gpu_end - cpu_after_gpu_start)
    total_time = total_end - total_start

    print("\n--- Timing Breakdown ---")
    print(f"CPU Time (reading + non-GPU processing): {cpu_time:.6f} s")
    if CUPY_AVAILABLE and gpu_total_transfer_start is not None:
        gpu_transfer_time = gpu_total_transfer_end - gpu_total_transfer_start
        gpu_compute_time = gpu_compute_end - gpu_compute_start
        print(f"GPU Time (H2D + compute + D2H): {gpu_transfer_time:.6f} s")
        print(f"  └─ GPU Compute only: {gpu_compute_time:.6f} s")
    else:
        print("GPU not used (CuPy not available); top-k & stats done on CPU.")
    print(f"\nExecution Time: {total_time:.6f} s")
    print("=====================================")


if __name__ == "__main__":
    main()


Using dataset: /content/TMDB_movie_dataset_v112.csv

=== GPU BLOCK-SIZE TUNING (Numba CUDA) ===
BLOCK_SIZE (threads_per_block) = 128
blocks_per_grid (auto from N)  = 8192
GPU kernel execution time (block tuning) = 0.172858 s
Sample scaled ratings (block): [12.5460005 12.6255    12.768     11.3595    11.565001 ]


=== GPU GRID-SIZE TUNING (Numba CUDA) ===


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


BLOCK_SIZE_GRID (threads_per_block) = 128
BLOCKS_PER_GRID (number of blocks)  = 8
GPU kernel execution time (grid tuning) = 0.099808 s
Sample scaled ratings (grid): [12.5460005 12.6255    12.768     11.3595    11.565001 ]

GPU Version – TMDB Movie Analysis
Movies processed: 1048575

Top 10 Movies by Revenue:
 1. "babben: the movie" | Revenue: 4999999999.00
 2. "Juliet and Romeo - 1231 ☆*: .｡. o(≧▽≦)o .｡.:*☆ UwU" | Revenue: 4999999999.00
 3. "Adventures in Bora Bora" | Revenue: 3000000000.00
 4. "TikTok Rizz Party" | Revenue: 3000000000.00
 5. "Bee Movie" | Revenue: 2930000000.00
 6. "Avatar" | Revenue: 2923706026.00
 7. "Avengers: Endgame" | Revenue: 2800000000.00
 8. "Avatar: The Way of Water" | Revenue: 2320250281.00
 9. "Titanic" | Revenue: 2264162353.00
10. "Star Wars: The Force Awakens" | Revenue: 2068223624.00

Top 10 Movies by Popularity:
 1. "Blue Beetle" | Year:2023 | Popularity: 2994.36
 2. "Gran Turismo" | Year:2023 | Popularity: 2680.59
 3. "A Female Boss  - Yua Mikami" | Y